## A simple Self-Attention, not trainable

This isn't trainable, just to demonstrate the attention mechanism.

In [2]:
import torch

# say we start with a few random embeddings, (5, 10) (num_embed, embed_size)

torch.manual_seed(42)
word_embs = torch.randn((5, 10))
word_embs

tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047,
         -0.7521,  1.6487],
        [-0.3925, -1.4036, -0.7279, -0.5594, -0.7688,  0.7624,  1.6423, -0.1596,
         -0.4974,  0.4396],
        [-0.7581,  1.0783,  0.8008,  1.6806,  1.2791,  1.2964,  0.6105,  1.3347,
         -0.2316,  0.0418],
        [-0.2516,  0.8599, -1.3847, -0.8712,  0.0780,  0.5258, -0.4880,  1.1914,
         -0.8140, -0.7360],
        [-0.8371, -0.9224, -0.0635,  0.6756, -0.0978,  1.8446, -1.1845,  1.3835,
         -1.2024,  0.7078]])

Then there comes dot products, simply, we are taking to dot product of every vector, with every other vector.

This is a demonstration of the 1st vector, with every other vector in x.

<img src="attachments/Screenshot 2026-02-05 at 2.46.47 PM.png" width="400px">

In [ ]:
query = word_embs[0] # the first vector

attn_scores = [] # empty attn score array right now

for index, embedding in enumerate(word_embs):
    attn_scores.append(torch.dot(query, embedding))

attn_scores = torch.tensor(attn_scores)
print(attn_scores)

tensor([19.0147, -2.5002, -5.3321, -1.7068, -6.9059])


a dot product really is just multiplying a vector with a vectors' tranpose, order doesn't matter

In [38]:
a = torch.rand(1, 5)
b = torch.rand(1, 5)

def dot_product(vec_a, vec_b):
    dot_result = vec_a @ vec_b.transpose(0, 1) # @ is vector/matrix multiplication in pytorch
    return dot_result
print(a)
print(b)
print(dot_product(a,b))


### we can even dig down further into element by element definition of it
def real_dot_product(vec_a, vec_b):

    sum = []
    for index in range(vec_a.size(1)):
        sum.append(vec_a[0, index] * vec_b[0, index])
    dot_product = torch.sum(torch.tensor(sum)).view(1,1)
    return dot_product

print(real_dot_product(a,b))

tensor([[0.9147, 0.2036, 0.2018, 0.2018, 0.9497]])
tensor([[0.6666, 0.9811, 0.0874, 0.0041, 0.1088]])
tensor([[0.9314]])
tensor([[0.9314]])


But, right now this is what we call the 'raw' attention values, you need to normalize them through an activation called softmax.

<img src="attachments/Screenshot 2026-02-05 at 3.05.33 PM.png" width="500px">

In the book, the diagram shows that the attention scores are computed first, and then normalized.

<img src="attachments/Screenshot 2026-02-05 at 4.21.21 PM.png" width="500px">

In [ ]:
def simple_softmax(x, dim=0):
    return torch.exp(x) / torch.exp(x).sum(dim=dim)


query = word_embs[0] # the first vector
attn_scores = torch.empty(word_embs.shape[0])

for index, emb in enumerate(word_embs):
    attn_scores[index] = dot_product(query.unsqueeze(0), emb.unsqueeze(0))

normalized_values = simple_softmax(attn_scores)
print(attn_scores)
print(attn_scores.sum())
print(normalized_values)
print(normalized_values.sum())

tensor([19.0147, -2.5002, -5.3321, -1.7068, -6.9059])
tensor(2.5696)
tensor([1.0000e+00, 4.5312e-10, 2.6688e-11, 1.0017e-09, 5.5311e-12])
tensor(1.)


let's do it on the example matrix in it's entirety.

In [45]:
def naive_attention(x):
    rows, cols = x.shape

    raw_scores = torch.empty((rows, rows), dtype=x.dtype)

    for row_index, query_embedding in enumerate(x):
        for col_index, key_embedding in enumerate(x):
            raw_scores[row_index, col_index] = dot_product(query_embedding.unsqueeze(0), key_embedding.unsqueeze(0)).item()
    
    raw_sums = torch.sum(raw_scores, dim=1)
    scores = simple_softmax(raw_scores, dim=1)
    sums = torch.sum(scores, dim=1)

    return raw_scores, raw_sums, scores, sums

rsc, rsu, sc, su = naive_attention(x)
print(rsc, '\n', rsu, '\n', '\n', sc, '\n', su)
print()
print("they don't add up to 1 due to numerical precision... problems")


tensor([[19.0147, -2.5002, -5.3321, -1.7068, -6.9059],
        [-2.5002,  7.3027, -1.8109, -0.1822,  1.5161],
        [-5.3321, -1.8109, 10.7298,  0.7764,  4.4225],
        [-1.7068, -0.1822,  0.7764,  6.6234,  2.5632],
        [-6.9059,  1.5161,  4.4225,  2.5632, 10.6884]]) 
 tensor([ 2.5696,  4.3255,  8.7857,  8.0740, 12.2843]) 
 
 tensor([[1.0000e+00, 5.5084e-05, 1.0558e-07, 2.3605e-04, 2.2798e-08],
        [4.5312e-10, 9.9622e-01, 3.5714e-06, 1.0843e-03, 1.0364e-04],
        [2.6688e-11, 1.0974e-04, 9.9813e-01, 2.8279e-03, 1.8958e-03],
        [1.0017e-09, 5.5939e-04, 4.7479e-05, 9.7897e-01, 2.9533e-04],
        [5.5311e-12, 3.0569e-03, 1.8197e-03, 1.6884e-02, 9.9771e-01]]) 
 tensor([1.0003, 0.9974, 1.0030, 0.9799, 1.0195])

they don't add up to 1 due to numerical precision... problems


As noted by the auther:

- The naive implementation above can suffer from numerical instability issues for large or small input values due to overflow and underflow issues
- Hence, in practice, it's recommended to use the PyTorch implementation of softmax instead, which has been highly optimized for performance:

In [47]:
attn_weights = torch.softmax(rsc, dim=1) # raw scores

print("Attention weights:", attn_weights)
print("Sum:", attn_weights.sum(dim=1))

Attention weights: tensor([[1.0000e+00, 4.5312e-10, 2.6688e-11, 1.0017e-09, 5.5311e-12],
        [5.5084e-05, 9.9622e-01, 1.0974e-04, 5.5939e-04, 3.0569e-03],
        [1.0558e-07, 3.5714e-06, 9.9813e-01, 4.7479e-05, 1.8197e-03],
        [2.3605e-04, 1.0843e-03, 2.8279e-03, 9.7897e-01, 1.6884e-02],
        [2.2798e-08, 1.0364e-04, 1.8958e-03, 2.9533e-04, 9.9771e-01]])
Sum: tensor([1., 1., 1., 1., 1.])


we aren't done yet, because these are just attention weights, not the attention result, we still need to multiply each input with the correspondoing attention weight.

as we've actually computed the entire attention weights matrix... I'll just get, say, the 2nd row out

In [ ]:
query = word_embs[1] # the second vector/input

context_vec_2 = torch.zeros(query.shape)
for index, emb_vector in enumerate(word_embs):
    context_vec_2 += attn_weights[1, index] * emb_vector

print(context_vec_2)

tensor([-0.3937, -1.4004, -0.7260, -0.5557, -0.7660,  0.7656,  1.6323, -0.1540,
        -0.4997,  0.4398])


## Attention Weights for All Input Tokens

Now, here's a cool trick, you tought we had to go through a double for loop and dot eveything with everything again...

In [ ]:
raw_scores = torch.zeros(word_embs.size(0), word_embs.size(0))

for row_index, query_embedding in enumerate(word_embs):
    for col_index, key_embedding in enumerate(word_embs):
        raw_scores[row_index, col_index] = dot_product(query_embedding.unsqueeze(0), key_embedding.unsqueeze(0)).item()

print(raw_scores)

tensor([[19.0147, -2.5002, -5.3321, -1.7068, -6.9059],
        [-2.5002,  7.3027, -1.8109, -0.1822,  1.5161],
        [-5.3321, -1.8109, 10.7298,  0.7764,  4.4225],
        [-1.7068, -0.1822,  0.7764,  6.6234,  2.5632],
        [-6.9059,  1.5161,  4.4225,  2.5632, 10.6884]])


Really you can just do a multiplication with one of the matrixes transposed to get the same effect of every vector dotting every vector!

In [ ]:
raw_scores = word_embs @ word_embs.transpose(1, 0)
print(raw_scores)

print()
norm_scores = torch.softmax(raw_scores, dim=1)
print(norm_scores)

tensor([[19.0147, -2.5002, -5.3321, -1.7068, -6.9059],
        [-2.5002,  7.3027, -1.8109, -0.1822,  1.5161],
        [-5.3321, -1.8109, 10.7298,  0.7764,  4.4225],
        [-1.7068, -0.1822,  0.7764,  6.6234,  2.5632],
        [-6.9059,  1.5161,  4.4225,  2.5632, 10.6884]])

tensor([[1.0000e+00, 4.5312e-10, 2.6688e-11, 1.0017e-09, 5.5311e-12],
        [5.5084e-05, 9.9622e-01, 1.0974e-04, 5.5939e-04, 3.0569e-03],
        [1.0558e-07, 3.5714e-06, 9.9813e-01, 4.7479e-05, 1.8197e-03],
        [2.3605e-04, 1.0843e-03, 2.8279e-03, 9.7897e-01, 1.6884e-02],
        [2.2798e-08, 1.0364e-04, 1.8958e-03, 2.9533e-04, 9.9771e-01]])


with these normalized scores, we can directly multiply it with the input embeddings, as that achieves the same effect.

In [67]:
context_vecs = norm_scores @ word_embs 
context_vecs

tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047,
         -0.7521,  1.6487],
        [-0.3937, -1.4004, -0.7260, -0.5557, -0.7660,  0.7656,  1.6323, -0.1540,
         -0.4997,  0.4398],
        [-0.7582,  1.0747,  0.7991,  1.6787,  1.2766,  1.2974,  0.6072,  1.3348,
         -0.2334,  0.0429],
        [-0.2625,  0.8281, -1.3549, -0.8379,  0.0777,  0.5501, -0.4942,  1.1929,
         -0.8186, -0.7076],
        [-0.8368, -0.9181, -0.0623,  0.6769, -0.0952,  1.8431, -1.1806,  1.3832,
         -1.2004,  0.7061]])

Now, is where the fun beings! If you've ever seen the original function of attention, it invloves 3 little buddies, K, Q, V placed in such a order.

<img src="attachments/Screenshot 2026-02-05 at 4.41.12 PM.png" width=500>

I've personally went over an analogy of what they are elsewhere... but the basic idea is that we need to introduce these learnable parameters into attention to produce better context_vectors.

- These three matrices are used to project the embedded input tokens, $x^{(i)}$, into query, key, and value vectors via matrix multiplication:

  - Query vector: $q^{(i)} = x^{(i)}\,W_q $
  - Key vector: $k^{(i)} = x^{(i)}\,W_k $
  - Value vector: $v^{(i)} = x^{(i)}\,W_v $


In [9]:
d_in = word_embs.size(1)
d_out = word_embs.size(1)

# I prefer linear layers over parameters
query_matrix = torch.nn.Linear(in_features=d_in, out_features=d_out, bias=False)
key_matrix = torch.nn.Linear(in_features=d_in, out_features=d_out, bias=False)
value_matrix = torch.nn.Linear(in_features=d_in, out_features=d_out, bias=False)

emb_2 = word_embs[1]

with torch.no_grad():
    q_2 = query_matrix(emb_2)
    k_2 = key_matrix(emb_2)
    v_2 = value_matrix(emb_2)

print(q_2)
print(k_2)
print(v_2)

tensor([ 0.0325,  0.1621, -0.4559, -0.7241,  0.8904, -0.2519, -0.4257,  0.2161,
         0.5191, -0.2646])
tensor([-0.7211,  0.1373,  0.7919,  0.0449, -0.4585,  0.4313, -0.1629, -0.4239,
        -0.0896,  0.0882])
tensor([-0.2047,  0.0548,  0.3097, -0.0959, -0.6914, -0.2218, -0.5215,  0.3965,
        -0.1788,  0.0752])


then it's just these 3 steps:

- compute unormalized attention scores
- normalize attention scores
- compute context vector

We might as well do it to the entire example

In [ ]:
# query, key, value 'values'
q = query_matrix(word_embs)
k = query_matrix(word_embs)
v = query_matrix(word_embs)


# unormalized attention scores
attn_scores = q @ k.transpose(0, 1)

# noramlize them
norm_attn_scores = torch.softmax(attn_scores / k.size(1)**0.5, dim=-1)

# compute context vector
with torch.no_grad():
    context_vecs = norm_attn_scores @ word_embs

context_vecs

tensor([[ 1.0972,  0.8917,  0.4541, -1.3630,  0.4450, -0.4674,  0.1067, -0.8044,
         -0.7167,  1.1393],
        [-0.1638, -0.1009, -0.2581, -0.2782,  0.0310,  0.7001,  0.3571,  0.3774,
         -0.6734,  0.3854],
        [-0.5055,  0.2858,  0.0883,  0.5727,  0.4865,  1.1157,  0.0545,  0.9928,
         -0.6291,  0.2326],
        [-0.3261,  0.1750, -0.3340, -0.0513,  0.1930,  0.8671, -0.0015,  0.7922,
         -0.7167,  0.1466],
        [-0.5367, -0.0880, -0.0782,  0.4055,  0.2323,  1.2262, -0.2471,  1.0117,
         -0.7977,  0.3393]])

## Turning Attention into a Class

This can all be written with a class

In [ ]:
import torch.nn as nn
from math import sqrt

class SelfAttention(nn.Module):
    
    def __init__(self, d_in, d_out):
        super().__init__()
        self.q = nn.Linear(d_in, d_out, bias=False)
        self.k = nn.Linear(d_in, d_out, bias=False)
        self.v = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):

        # x dim = [batch_size, text_length, emb_size]

        q_x = self.q(x)
        k_x = self.k(x)
        v_x = self.v(x)

        scores = (q_x @ k_x.transpose(-2, -1)) / sqrt(k_x.shape[-1])  # [batch_size, text_length, text_length]
        attn = torch.softmax(scores, dim=-1)             # softmax over keys
        out = attn @ v_x                                   # [batch_size, text_length, d_out]
        return out
    
    def attn(self, x):

        q_x = self.q(x)
        k_x = self.k(x)

        scores = (q_x @ k_x.transpose(-2, -1)) / sqrt(k_x.shape[-1])  # [batch_size, text_length, text_length]
        return scores

Urgh, let's use their example now

<img src="attachments/Screenshot 2026-02-09 at 9.23.41 PM.png" width=500>

In [21]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

inputs.shape


torch.Size([6, 3])

In [22]:
torch.manual_seed(42)

d_in = inputs.shape[1]
d_out = 2 # they used this number in the book

self_attn = SelfAttention(d_in, d_out)
with torch.no_grad():
  outputs = self_attn(inputs)

print(outputs.shape)
outputs

torch.Size([6, 2])


tensor([[0.3755, 0.2777],
        [0.3761, 0.2831],
        [0.3761, 0.2833],
        [0.3768, 0.2763],
        [0.3754, 0.2836],
        [0.3772, 0.2746]])

### Masked Attention

The simplest way to make masked attention is by created a mask through Pytorch's tril function.

In [ ]:
attn_scores = self_attn.attn(inputs)
attn_scores

tensor([[ 0.0359,  0.1084,  0.1016,  0.0819, -0.0501,  0.1509],
        [ 0.1525,  0.2424,  0.2295,  0.1648, -0.0673,  0.2970],
        [ 0.1530,  0.2451,  0.2321,  0.1670, -0.0689,  0.3011],
        [ 0.0889,  0.1272,  0.1207,  0.0842, -0.0298,  0.1509],
        [ 0.1179,  0.2258,  0.2130,  0.1597, -0.0776,  0.2902],
        [ 0.0897,  0.1094,  0.1043,  0.0691, -0.0175,  0.1224]],
       grad_fn=<DivBackward0>)

In [ ]:
context_len = attn_scores.shape[1]
mask = torch.tril(torch.ones((context_len, context_len)))
mask

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

Now we just need to perform element by element multiplication from the mask to our attn_weights.

It's real name is called a hadamard product, but... who cares, it's just a multiplication symbol in C.

In [ ]:
masked_attn_scores = attn_scores * mask # applying mask
print(masked_attn_scores)

tensor([[ 0.0359,  0.0000,  0.0000,  0.0000, -0.0000,  0.0000],
        [ 0.1525,  0.2424,  0.0000,  0.0000, -0.0000,  0.0000],
        [ 0.1530,  0.2451,  0.2321,  0.0000, -0.0000,  0.0000],
        [ 0.0889,  0.1272,  0.1207,  0.0842, -0.0000,  0.0000],
        [ 0.1179,  0.2258,  0.2130,  0.1597, -0.0776,  0.0000],
        [ 0.0897,  0.1094,  0.1043,  0.0691, -0.0175,  0.1224]],
       grad_fn=<MulBackward0>)


However, if the two tensors do not have the same shape, PyTorch broadcasts dimensions of size 1 are automatically expanded to match the corresponding dimension of the other tensor, allowing element-wise multiplication to proceed.

In [33]:
# making sure they sum to 1 (informal softmax)
row_sums = masked_attn_scores.sum(dim=-1, keepdim=True)
masked_norm_scores = masked_attn_scores / row_sums
print(masked_norm_scores)

tensor([[ 1.0000,  0.0000,  0.0000,  0.0000, -0.0000,  0.0000],
        [ 0.3862,  0.6138,  0.0000,  0.0000, -0.0000,  0.0000],
        [ 0.2427,  0.3890,  0.3683,  0.0000, -0.0000,  0.0000],
        [ 0.2111,  0.3021,  0.2867,  0.2001, -0.0000,  0.0000],
        [ 0.1846,  0.3535,  0.3335,  0.2500, -0.1216,  0.0000],
        [ 0.1879,  0.2292,  0.2184,  0.1448, -0.0366,  0.2564]],
       grad_fn=<DivBackward0>)


This is a bit wasteful, and we can use the property of the softmax, with a bit of changes to our masking tensor to optimize this masking process.

<img src="attachments/Screenshot 2026-02-09 at 9.53.34 PM.png" width=500>

In [40]:
keys = self_attn.k(inputs)
attn_scores = self_attn.attn(inputs)

mask = torch.triu(torch.ones((context_len, context_len)), diagonal=1)
inf_masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(inf_masked)

attn_weights = torch.softmax(inf_masked / sqrt(keys.shape[-1]), dim=-1)
attn_weights

tensor([[ 0.0359,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.1525,  0.2424,    -inf,    -inf,    -inf,    -inf],
        [ 0.1530,  0.2451,  0.2321,    -inf,    -inf,    -inf],
        [ 0.0889,  0.1272,  0.1207,  0.0842,    -inf,    -inf],
        [ 0.1179,  0.2258,  0.2130,  0.1597, -0.0776,    -inf],
        [ 0.0897,  0.1094,  0.1043,  0.0691, -0.0175,  0.1224]],
       grad_fn=<MaskedFillBackward0>)


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4841, 0.5159, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3200, 0.3416, 0.3384, 0.0000, 0.0000, 0.0000],
        [0.2471, 0.2539, 0.2527, 0.2463, 0.0000, 0.0000],
        [0.1980, 0.2137, 0.2118, 0.2040, 0.1725, 0.0000],
        [0.1678, 0.1701, 0.1695, 0.1653, 0.1555, 0.1717]],
       grad_fn=<SoftmaxBackward0>)

## Dropout

We know the good old freiend dropout, it's a technique that's applicable everywhere in neural networks, and this includes in attention.

There are in fact 2 different places to possibly place this mechamism, one after calculating the attention weights, or after applying the attention weights to the 'value' of the embeddings.

In [43]:
example_tensor = torch.tril(torch.ones(6, 6))

dropout = torch.nn.Dropout(0.5)

print(example_tensor)
print()
print(dropout(example_tensor))

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

tensor([[0., 0., 0., 0., 0., 0.],
        [0., 2., 0., 0., 0., 0.],
        [2., 0., 2., 0., 0., 0.],
        [0., 2., 2., 0., 0., 0.],
        [0., 2., 2., 0., 0., 0.],
        [2., 2., 2., 2., 0., 0.]])


We will be applying this to the masked attention values.

<img src="attachments/Screenshot 2026-02-09 at 10.12.50 PM.png" width=500>

In [45]:
keys = self_attn.k(inputs)
attn_scores = self_attn.attn(inputs)

mask = torch.triu(torch.ones((context_len, context_len)), diagonal=1)
inf_masked = attn_scores.masked_fill(mask.bool(), -torch.inf)

attn_weights = torch.softmax(inf_masked / sqrt(keys.shape[-1]), dim=-1)
dropout_attn_weights = dropout(attn_weights)

print(attn_weights)

dropout_attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4841, 0.5159, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3200, 0.3416, 0.3384, 0.0000, 0.0000, 0.0000],
        [0.2471, 0.2539, 0.2527, 0.2463, 0.0000, 0.0000],
        [0.1980, 0.2137, 0.2118, 0.2040, 0.1725, 0.0000],
        [0.1678, 0.1701, 0.1695, 0.1653, 0.1555, 0.1717]],
       grad_fn=<SoftmaxBackward0>)


tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9682, 1.0318, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.4926, 0.0000, 0.0000],
        [0.3961, 0.4275, 0.0000, 0.0000, 0.3449, 0.0000],
        [0.0000, 0.3403, 0.0000, 0.0000, 0.3111, 0.0000]],
       grad_fn=<MulBackward0>)

Dropout has scaled some of the values by 2 since it dropped 50% of the values.

Vanishing gradients, explodin gradients...

## Implementation of Updated Self Attention Class

In [65]:
class self_attn_v2(nn.Module):

    def __init__(self, emb_size, d_out, context_window, dropout):
        super().__init__()
        self.q = nn.Linear(emb_size, d_out, bias=False)
        self.k = nn.Linear(emb_size, d_out, bias=False)
        self.v = nn.Linear(emb_size, d_out, bias=False)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones((context_window, context_window)), diagonal=1)) # this is new

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        q_x = self.q(x)
        k_x = self.k(x)
        v_x = self.v(x)

        attn_scores = q_x @ k_x.transpose(-2, -1)
        attn_scores = attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / sqrt(k_x.shape[1]), dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ v_x
        return context_vec, attn_weights

In [66]:
torch.manual_seed(123)

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # 2 inputs with 6 tokens each, and each token has embedding dimension 3

context_window = batch.shape[1]
attn = self_attn_v2(d_in, d_out, context_window, 0.5)

context_vecs, attn_weights = attn(batch)

print(context_vecs)
print(context_vecs.shape)

torch.Size([2, 6, 3])
tensor([[[-0.9038,  0.4432],
         [-0.4432,  0.2173],
         [-0.4808, -0.1330],
         [-0.5829,  0.0099],
         [-0.6206, -0.0524],
         [-0.1412, -0.0503]],

        [[ 0.0000,  0.0000],
         [-1.1712,  0.0175],
         [-0.7746,  0.0111],
         [-0.9097, -0.2758],
         [-0.7610, -0.0715],
         [-0.6761, -0.0976]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([2, 6, 2])


Now, since we suddenly introduced 2 new things, I think we should explain what they are.

1. registor buffer
2. masked fill

Registor Buffer:

You sometimes need tensors that are:
- part of the model
- move with the model to GPU / CPU
- saved and loaded with the model, but shouldn't be trained

An attention mask is the textbook example, and that's what the registor buffer does in Pytorch.

Masked Fill:

You want to disable certain elements in a tensor before softmax.

In attention, that means:
- future tokens
- padding tokens
- anything the model should not attend to

## From Single to Multiple Head Attention

Multi-head attention isn't anything crazy, the idea simply is that instad of just 1 set of q, k, v matrixes, we have multipel sets of them acting on the same input embeddings.

Technically not mutiple sets... we just divide one q/k/v matrix into multiple heads

The result is concaconated, head to tail.

In [9]:
import torch
from math import sqrt
from torch import nn as nn

In [15]:
class MultAttn(nn.Module):

    def __init__(self, d_in=64, d_out=64, context_length=10, dropout=0.5, num_heads=8):
        super().__init__()
        self.w_q = nn.Linear(d_in, d_out, bias=False)
        self.w_k = nn.Linear(d_in, d_out, bias=False)
        self.w_v = nn.Linear(d_in, d_out, bias=False)
        self.head_dim = d_out // num_heads
        self.num_heads = num_heads

        self.dropout = nn.Dropout(dropout)
        self.mlp = nn.Linear(d_out, d_out)

        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):

        # x shape: [batch_size, num_tokens, d_in] d_in = emb_size
        b, num_tokens, d_in = x.shape

        # getting qkv
        q = self.w_q(x) # [batch_size, num_tokens, d_out]
        k = self.w_k(x)
        v = self.w_v(x)

        # reshape, multi-head attention
        # this didn't add anything new! We just 'split' the qkv (d_out) into 2 dims, (num_heads, head_size)
        q = q.view(b, num_tokens, self.num_heads, self.head_dim)
        k = k.view(b, num_tokens, self.num_heads, self.head_dim)
        v = v.view(b, num_tokens, self.num_heads, self.head_dim)

        # But we don't want the operations on num_heads! We want to do attention for each head respectively
        q = q.reshape(b, self.num_heads, num_tokens, self.head_dim)
        k = k.reshape(b, self.num_heads, num_tokens, self.head_dim)
        v = v.reshape(b, self.num_heads, num_tokens, self.head_dim)

        # doing attention
        attn_scores = q @ k.transpose(-2, -1) # [b, num_heads, num_tokens, num_tokens]

        # masking
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / sqrt(k.shape[-1]),dim=-1)

        # dropuot
        attn_weights = self.dropout(attn_weights)

        # combine heads after multiply with value
        # [b, num_heads, num_tokens, head_size] = [b, num_heads, num_tokens, num_tokens] * [b, num_heads, num_tokens, head_size] 
        context_vec = attn_weights @ v

        # [b, num_tokens, num_heads, head_size], this is for concatonation
        context_vec = context_vec.transpose(1, 2)

        # concat (which means reshaping here)
        # cintguous helps us change where the arrays are stored on a matrix, making it continous
        context_vec = context_vec.contiguous().view(b, num_tokens, d_in)
        context_vec = self.mlp(context_vec) # optional projection

        return context_vec
        

In [20]:
# [batch_size, num_tokens, emb_size]
rand_batch = torch.rand([4, 3, 8])
mult_attn = MultAttn(d_in=8, d_out=8, num_heads=2)

context_vecs = mult_attn(rand_batch)
print(context_vecs.shape)
context_vecs

torch.Size([4, 3, 8])


tensor([[[ 0.2672, -0.3366, -0.0572, -0.6155, -0.1392,  0.4272,  0.0556,
           0.6815],
         [ 0.4157, -0.2139,  0.1998, -0.5992, -0.4919,  0.2230, -0.0418,
           0.4798],
         [ 0.3902, -0.3676, -0.0125, -0.6549, -0.3516,  0.1741, -0.1425,
           0.4437]],

        [[ 0.2378, -0.3151,  0.0242, -0.5689, -0.1909,  0.3376,  0.0753,
           0.6498],
         [ 0.2487, -0.3723,  0.0425, -0.6543, -0.3826,  0.0563, -0.3726,
           0.3060],
         [ 0.2627, -0.3645,  0.1558, -0.6500, -0.4932,  0.0679, -0.0143,
           0.5921]],

        [[ 0.4016, -0.1969,  0.3051, -0.2412, -0.3502,  0.2372, -0.0626,
           0.1616],
         [ 0.3226, -0.0620,  0.3154, -0.5250, -0.3003,  0.1921,  0.1089,
           0.4930],
         [ 0.5546, -0.3236,  0.1210, -0.4347, -0.5030,  0.1251,  0.1491,
           0.4561]],

        [[ 0.2961, -0.1664,  0.3473, -0.2412, -0.2856,  0.2615, -0.1671,
           0.2219],
         [ 0.2991, -0.3023,  0.1609, -0.3321, -0.2127,  0.3317, 